In [1]:
import os
import re
import glob
import json
import subprocess
import numpy as np
import pandas as pd
import torch

In [2]:
INPUT = "/kaggle/input/datasets/nocturnalnerd18"
USERNAME = "nocturnalnerd18"
TYPES = ["attribute", "relation"]
CONTROL_RATIO = 1.5
MIN_CONTROLS = 300
SHARES = {"true_yes": 0.6, "true_no": 0.4}
PLACEHOLDER_SEC_PER_QUESTION = 3.0


def batch_number(path):
    return int(re.search(r"_batch_(\d+)\.pt$", path).group(1))


with open(f"{INPUT}/vlm-questions/all_questions.json") as f:
    all_questions = json.load(f)
assert len(all_questions) == 17492, "attach vlm-questions version 2"
question_by_id = {q["question_id"]: q for q in all_questions}

with open(f"{INPUT}/vlm-labeled-examples/labeled_examples.json") as f:
    labels = pd.DataFrame(json.load(f))
assert "cache" in labels.columns, "attach vlm-labeled-examples version 3"
splits = labels[labels["cache"] == "question_aware"][["question_id", "split"]].drop_duplicates("question_id")

rows = []
for t in TYPES:
    paths = sorted(glob.glob(f"{INPUT}/dla-qwen/{t}_batch_*.pt"), key=batch_number)
    assert paths, f"attach dla-qwen ({t} batches missing)"
    for path in paths:
        for e in torch.load(path):
            rows.append({
                "question_id": e["question_id"], "type": e["hallucination_type"], "prefix": e["prefix"],
                "logit_diff": float(e["logit_diff"]), "in_pair": bool(e["in_pair"]),
            })
rec = pd.DataFrame(rows)
assert rec.question_id.is_unique
rec["source"] = rec.prefix.str.rstrip("_")
n_all = len(rec)
rec = rec.merge(splits, on="question_id", how="inner")
assert len(rec) > 0.99 * n_all, "most dla-qwen entries have no split"
print(n_all, "dla-qwen entries,", len(rec), "joined to a split")

print(pd.crosstab([rec.type, rec.source], rec.in_pair, normalize="index").round(4))
rec = rec[rec.in_pair].copy()
rec["truth"] = rec.question_id.map(lambda q: question_by_id[q]["ground_truth_answer"].strip().lower())
rec["model_answer"] = np.where(rec.logit_diff > 0, "yes", "no")
rec["group"] = np.select(
    [
        (rec.truth == "yes") & (rec.model_answer == "yes"),
        (rec.truth == "no") & (rec.model_answer == "no"),
        (rec.truth == "no") & (rec.model_answer == "yes"),
        (rec.truth == "yes") & (rec.model_answer == "no"),
    ],
    ["true_yes", "true_no", "false_yes", "false_no"],
    default="other",
)
assert (rec.group != "other").all()
print(len(rec), "questions with the top-1 token in the yes/no pair")
print(pd.crosstab([rec.type, rec.source, rec.split], rec.group))

14134 dla-qwen entries, 14126 joined to a split
in_pair              False   True 
type      source                  
attribute amber     0.1157  0.8843
relation  amber     0.0559  0.9441
          reefknot  0.0442  0.9558
12937 questions with the top-1 token in the yes/no pair
group                     false_no  false_yes  true_no  true_yes
type      source   split                                        
attribute amber    test        133         11      512       342
                   train       656         89     2365      1624
                   val         126         13      507       360
relation  amber    test         91          0       97        42
                   train       392          2      480       232
                   val          88          2      104        41
          reefknot test         80        152      202       266
                   train       329        697      927      1270
                   val          64        153      201       287


In [3]:
parts = []
for t in TYPES:
    d = rec[rec.type == t]
    fy = d[d.group == "false_yes"]
    parts.append(fy.assign(stratum="false_yes"))
    budget = max(int(CONTROL_RATIO * len(fy)), MIN_CONTROLS)
    for g, share in SHARES.items():
        pool = d[d.group == g]
        if len(pool) == 0:
            continue
        quota = min(int(budget * share), len(pool))
        parts.append(pool.groupby(["source", "split"], group_keys=False).sample(frac=quota / len(pool), random_state=0).assign(stratum=g))

selection = pd.concat(parts).sample(frac=1.0, random_state=0).reset_index(drop=True)
assert selection.question_id.is_unique
assert (selection.stratum == "false_yes").sum() == int((rec.group == "false_yes").sum()), "false_yes rows were dropped"

print(pd.crosstab([selection.type, selection.source, selection.split], selection.stratum, margins=True))
print("selected per type:", selection.groupby("type").size().to_dict(), "of", rec.groupby("type").size().to_dict())
print(f"{len(selection)} questions at a placeholder {PLACEHOLDER_SEC_PER_QUESTION} s/question: about "
      f"{len(selection) * PLACEHOLDER_SEC_PER_QUESTION / 3600:.1f} h of collection (replace with the measured speed)")

stratum                   false_yes  true_no  true_yes   All
type      source   split                                    
attribute amber    test          11       18        26    55
                   train         89       84       126   299
                   val           13       18        28    59
relation  amber    test           0       29        18    47
                   train          2      144        98   244
                   val            2       31        17    50
          reefknot test         152       61       113   326
                   train        697      278       538  1513
                   val          153       60       121   334
All                            1119      723      1085  2927
selected per type: {'attribute': 413, 'relation': 2514} of {'attribute': 6738, 'relation': 6199}
2927 questions at a placeholder 3.0 s/question: about 2.4 h of collection (replace with the measured speed)


In [4]:
SEL_DIR = "/kaggle/working/atp_qwen_selection"
os.makedirs(SEL_DIR, exist_ok=True)

with open(os.path.join(SEL_DIR, "selection.json"), "w") as f:
    json.dump({
        "selected": {t: selection[selection.type == t].question_id.tolist() for t in TYPES},
        "stratum": dict(zip(selection.question_id, selection.stratum)),
        "group": dict(zip(selection.question_id, selection.group)),
        "source": dict(zip(selection.question_id, selection.source)),
        "split": dict(zip(selection.question_id, selection.split)),
        "prefix": dict(zip(selection.question_id, selection.prefix)),
    }, f)

with open(os.path.join(SEL_DIR, "dataset-metadata.json"), "w") as f:
    json.dump({
        "title": "atp-qwen-selection",
        "id": f"{USERNAME}/atp-qwen-selection",
        "subtitle": "Question subsample for Qwen2.5-VL attribution patching",
        "description": "Questions selected for Qwen2.5-VL-7B attribution patching from the dla-qwen answers (top-1 token in the yes/no pair): "
                       "every false_yes row (model said yes, truth no) plus stratified true_yes (60%) and true_no (40%) controls by source and split, "
                       "about 1.5 controls per false_yes row for each of attribute and relation. "
                       "stratum, group, source, split and prefix map each question_id to its selection group, label group, dataset, split and answer-token pair.",
        "licenses": [{"name": "CC0-1.0"}],
    }, f)


def _run(cmd):
    r = subprocess.run(cmd, capture_output=True, text=True)
    out = (r.stdout + r.stderr).strip()
    return r.returncode == 0 and "error" not in out.lower(), out


ok, out = _run(["kaggle", "datasets", "create", "-p", SEL_DIR])
if not ok:
    ok, out2 = _run(["kaggle", "datasets", "version", "-p", SEL_DIR, "-m", "selection"])
    out += "\n" + out2
print(out)
assert ok, "dataset upload failed"

Starting upload for file selection.json
Upload successful: selection.json (504KB)
Your private Dataset is being created. Please check progress at https://www.kaggle.com/datasets/nocturnalnerd18/atp-qwen-selection

  0%|          | 0.00/504k [00:00<?, ?B/s]
100%|██████████| 504k/504k [00:00<00:00, 2.10MB/s]
